In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader, random_split
from transformers import ViTForImageClassification
from sklearn.metrics import accuracy_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# 64x64 transforms for Simple CNN and HOG+SVM
transform_train_64 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_64 = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 224x224 transforms for ResNet and ViT
transform_train_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Download all datasets
train_full_64 = datasets.GTSRB(root='./data', split='train', transform=transform_train_64, download=True)
train_full_224 = datasets.GTSRB(root='./data', split='train', transform=transform_train_224, download=False)
test_64 = datasets.GTSRB(root='./data', split='test', transform=transform_64, download=True)
test_224 = datasets.GTSRB(root='./data', split='test', transform=transform_224, download=False)

# Splits
train_size = int(0.9 * len(train_full_64))
val_size = len(train_full_64) - train_size

train_64, val_64 = random_split(train_full_64, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_224, val_224 = random_split(train_full_224, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Loaders
train_loader_64 = DataLoader(train_64, batch_size=64, shuffle=True, num_workers=2)
val_loader_64 = DataLoader(val_64, batch_size=64, shuffle=False, num_workers=2)
test_loader_64 = DataLoader(test_64, batch_size=64, shuffle=False, num_workers=2)
train_loader_224 = DataLoader(train_224, batch_size=64, shuffle=True, num_workers=2)
val_loader_224 = DataLoader(val_224, batch_size=64, shuffle=False, num_workers=2)
test_loader_224 = DataLoader(test_224, batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_64)}, Val: {len(val_64)}, Test: {len(test_64)}")
print("All datasets ready!")

100%|██████████| 89.0M/89.0M [00:07<00:00, 12.4MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 271kB/s]


Train: 23976, Val: 2664, Test: 12630
All datasets ready!


In [4]:
def apply_gaussian_blur(image, severity=1):
    kernel_sizes = [3, 5, 7, 9, 11]
    ksize = kernel_sizes[severity - 1]
    img_array = np.array(image)
    blurred = cv2.GaussianBlur(img_array, (ksize, ksize), 0)
    return Image.fromarray(blurred)

def apply_noise(image, severity=1):
    noise_levels = [5, 10, 20, 35, 50]
    noise_std = noise_levels[severity - 1]
    img_array = np.array(image).astype(np.float32)
    noise = np.random.normal(0, noise_std, img_array.shape)
    noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def apply_brightness(image, severity=1):
    brightness_factors = [0.8, 0.6, 0.4, 0.2, 0.1]
    factor = brightness_factors[severity - 1]
    img_array = np.array(image).astype(np.float32)
    darkened = np.clip(img_array * factor, 0, 255).astype(np.uint8)
    return Image.fromarray(darkened)

def apply_fog(image, severity=1):
    fog_intensities = [0.2, 0.35, 0.5, 0.65, 0.8]
    fog_intensity = fog_intensities[severity - 1]
    img_array = np.array(image).astype(np.float32)
    fog_layer = np.ones_like(img_array) * 255
    fogged = img_array * (1 - fog_intensity) + fog_layer * fog_intensity
    fogged = np.clip(fogged, 0, 255).astype(np.uint8)
    return Image.fromarray(fogged)

print("Degradation functions defined!")
print("Blur: severity 1-3")
print("Noise: severity 1-3")
print("Brightness: severity 1-4")
print("Fog: severity 1-4")

Degradation functions defined!
Blur: severity 1-3
Noise: severity 1-3
Brightness: severity 1-4
Fog: severity 1-4


In [5]:
# Extract HOG features
def extract_hog_features(dataset):
    features = []
    labels = []
    for i in tqdm(range(len(dataset)), desc="Extracting HOG features"):
        image, label = dataset[i]
        image_np = image.permute(1, 2, 0).numpy()
        hog_features = hog(
            image_np,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            channel_axis=-1
        )
        features.append(hog_features)
        labels.append(label)
    return np.array(features), np.array(labels)

print("Extracting HOG features for training set...")
X_train_hog, y_train_hog = extract_hog_features(train_64)

print("Extracting HOG features for test set...")
X_test_hog, y_test_hog = extract_hog_features(test_64)

print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hog)
X_test_scaled = scaler.transform(X_test_hog)

print("Training HOG+SVM...")
svm_model = LinearSVC(C=0.1, max_iter=2000, random_state=42)
svm_model.fit(X_train_scaled, y_train_hog)

y_pred = svm_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test_hog, y_pred)
print(f"HOG+SVM Test Accuracy: {accuracy * 100:.2f}%")

Extracting HOG features for training set...


Extracting HOG features: 100%|██████████| 23976/23976 [01:01<00:00, 391.90it/s]


Extracting HOG features for test set...


Extracting HOG features: 100%|██████████| 12630/12630 [00:29<00:00, 422.70it/s]


Scaling features...
Training HOG+SVM...
HOG+SVM Test Accuracy: 78.36%
